In [25]:
import requests
import json

def print_keys(data, path=""):
    """
    Recursively finds and prints all keys in a nested dictionary/list.
    """
    if isinstance(data, dict):
        for key, value in data.items():
            # Build the path string (e.g., "user -> profile -> name")
            new_path = f"{path} -> {key}" if path else key
            print(new_path)
            # Dive deeper
            print_keys(value, new_path)
            
    elif isinstance(data, list):
        for index, item in enumerate(data):
            # We don't usually name list indices as 'keys', 
            # but we need to check if the items inside have keys.
            print_keys(item, f"{path}[{index}]")

# Usage:
# data = response.json()
# print_keys(data)

url = "https://api.elections.kalshi.com/trade-api/v2/events/KXMARMAD-26"
params = {
    'with_nested_markets': 'true',
}

response = requests.get(url, params) # get(url, params) where params = {'...':'value'}
data = response.json()
print_keys(data)
# print(json.dumps(data,indent=4))

event
event -> available_on_brokers
event -> category
event -> collateral_return_type
event -> event_ticker
event -> last_updated_ts
event -> markets
event -> markets[0] -> can_close_early
event -> markets[0] -> close_time
event -> markets[0] -> created_time
event -> markets[0] -> custom_strike
event -> markets[0] -> custom_strike -> basketball_team
event -> markets[0] -> early_close_condition
event -> markets[0] -> event_ticker
event -> markets[0] -> expected_expiration_time
event -> markets[0] -> expiration_time
event -> markets[0] -> expiration_value
event -> markets[0] -> fractional_trading_enabled
event -> markets[0] -> last_price_dollars
event -> markets[0] -> latest_expiration_time
event -> markets[0] -> liquidity_dollars
event -> markets[0] -> market_type
event -> markets[0] -> no_ask_dollars
event -> markets[0] -> no_bid_dollars
event -> markets[0] -> no_sub_title
event -> markets[0] -> notional_value_dollars
event -> markets[0] -> open_interest_fp
event -> markets[0] -> open_

In [ ]:
import asyncio
import httpx
import pandas as pd
from tqdm.asyncio import tqdm

# Config
EVENTS = ["KXMARMAD-25", "KXMARMAD-26"]
BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"

async def get_team_mapping():
    mapping_data = []
    
    async with httpx.AsyncClient() as client:
        for event_ticker in EVENTS:
            print(f"Resolving teams for {event_ticker}...")
            
            # Check both Market Discovery endpoints
            for path in ["/markets", "/historical/markets"]:
                params = {"event_ticker": event_ticker, "limit": 1000}
                
                # Handling pagination for market discovery
                cursor = None
                while True:
                    if cursor: params["cursor"] = cursor
                    
                    resp = await client.get(f"{BASE_URL}{path}", params=params)
                    if resp.status_code != 200:
                        break
                    
                    data = resp.json()
                    markets = data.get('markets', [])
                    
                    for m in markets:
                        # Priority: yes_sub_title -> no_sub_title -> title
                        team_name = (
                            m.get('yes_sub_title') or 
                            m.get('no_sub_title') or 
                            m.get('title')
                        )
                        
                        mapping_data.append({
                            "ticker": m.get('ticker'),
                            "event": event_ticker,
                            "team_name": team_name,
                            "market_type": "historical" if "historical" in path else "live"
                        })
                    
                    cursor = data.get('cursor')
                    if not cursor or not markets:
                        break
                        
    return pd.DataFrame(mapping_data)

# Jupyter execution
team_mapping_df = await get_team_mapping()

# Clean up: Often the sub_title has extra whitespace or "Yes/No" artifacts
team_mapping_df['team_name'] = team_mapping_df['team_name'].str.strip()

print(f"Mapped {len(team_mapping_df)} tickers to team names.")
team_mapping_df.tail(10)

Resolving teams for KXMARMAD-25...
Resolving teams for KXMARMAD-26...
Mapped 181 tickers to team names.


,ticker,event,team_name,market_type
0,KXMARMAD-25-HP,KXMARMAD-25,High Point,historical
1,KXMARMAD-25-NEOM,KXMARMAD-25,Omaha,historical
2,KXMARMAD-25-OKLA,KXMARMAD-25,Oklahoma,historical
3,KXMARMAD-25-WOF,KXMARMAD-25,Wofford,historical
4,KXMARMAD-25-SFPA,KXMARMAD-25,St. Francis (PA),historical
5,KXMARMAD-25-VCU,KXMARMAD-25,VCU,historical
6,KXMARMAD-25-SIUE,KXMARMAD-25,SIU Edwardsville,historical
7,KXMARMAD-25-LIB,KXMARMAD-25,Liberty,historical
8,KXMARMAD-25-BAY,KXMARMAD-25,Baylor,historical
9,KXMARMAD-25-ARK,KXMARMAD-25,Arkansas,historical


In [35]:
team_mapping_df.to_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/market_data_store/name_maping.csv')

In [6]:
import requests
import json

# Fetch data
hurl = "https://api.elections.kalshi.com/trade-api/v2/historical/trades?limit=500&ticker=KXMARMAD-25-DU"
hresponse = requests.get(hurl)

url = "https://api.elections.kalshi.com/trade-api/v2/markets/trades?limit=500&ticker=KXMARMAD-25-DU"
response = requests.get(url)

# 1. Extract JSON data
h_data = hresponse.json()
data = response.json()

def compare_structures(dict1, dict2, name1="Historical", name2="Live"):
    print(f"--- Structural Comparison: {name1} vs {name2} ---")
    
    # Compare Top Level Keys
    keys1 = set(dict1.keys())
    keys2 = set(dict2.keys())
    
    print(f"Common keys: {keys1 & keys2}")
    if keys1 - keys2:
        print(f"Keys only in {name1}: {keys1 - keys2}")
    if keys2 - keys1:
        print(f"Keys only in {name2}: {keys2 - keys1}")
    
    # Compare keys of the first item in 'trades' list (if exists)
    list_key = 'trades' # Kalshi usually wraps results in a 'trades' key
    if list_key in dict1 and list_key in dict2:
        if dict1[list_key] and dict2[list_key]:
            t1_keys = set(dict1[list_key][0].keys())
            t2_keys = set(dict2[list_key][0].keys())
            
            print(f"\n--- Individual Trade Object Keys ---")
            print(f"Common trade keys: {t1_keys & t2_keys}")
            if t1_keys - t2_keys:
                print(f"Trade keys only in {name1}: {t1_keys - t2_keys}")
            if t2_keys - t1_keys:
                print(f"Trade keys only in {name2}: {t2_keys - t1_keys}")
        else:
            print(f"\nOne of the 'trades' lists is empty.")

# Run comparison
compare_structures(h_data, data)

# 2. Check if the entire data payload is identical
print(f"\nAre payloads identical? {h_data == data}")

# 3. Deep Dive into the differences
h_trades = h_data.get('trades', [])
l_trades = data.get('trades', [])

# Convert trade lists to IDs for comparison
h_ids = {t['trade_id'] for t in h_trades}
l_ids = {t['trade_id'] for t in l_trades}

print(f"--- Data Content Deep Dive ---")
print(f"Total Historical trades: {len(h_trades)}")
print(f"Total Live trades: {len(l_trades)}")

# Check for ID overlap
common_ids = h_ids & l_ids
print(f"Number of identical Trade IDs in both: {len(common_ids)}")

if len(common_ids) > 0:
    # Pick one common trade and see if the values match
    sample_id = list(common_ids)[0]
    h_sample = next(t for t in h_trades if t['trade_id'] == sample_id)
    l_sample = next(t for t in l_trades if t['trade_id'] == sample_id)
    
    if h_sample == l_sample:
        print(f"Result: The trades with the same ID have identical data.")
    else:
        print(f"Result: Found a trade (ID: {sample_id}) where values differ!")
        # Print the specific field differences
        for key in h_sample:
            if h_sample[key] != l_sample[key]:
                print(f"  Field '{key}': Historical={h_sample[key]}, Live={l_sample[key]}")

# Check for unique trades
print(f"Trades only in Historical: {len(h_ids - l_ids)}")
print(f"Trades only in Live: {len(l_ids - h_ids)}")

--- Structural Comparison: Historical vs Live ---
Common keys: {'cursor', 'trades'}

One of the 'trades' lists is empty.

Are payloads identical? False
--- Data Content Deep Dive ---
Total Historical trades: 500
Total Live trades: 0
Number of identical Trade IDs in both: 0
Trades only in Historical: 500
Trades only in Live: 0


In [10]:
import asyncio
import httpx
import pandas as pd
from tqdm.asyncio import tqdm

# Config
EVENTS = ["KXMARMAD-25", "KXMARMAD-26"]
BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
CONCURRENCY_LIMIT = 5 # Lower is safer for laptop RAM and Rate Limits

async def fetch_with_backoff(client, url, params, ticker_label=""):
    """Fetch a single page with retries on rate limits (429)."""
    for attempt in range(5): # Up to 5 retries
        try:
            resp = await client.get(url, params=params, timeout=30.0)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait = (2 ** attempt) + 1
                # print(f"  [Rate Limit] {ticker_label} waiting {wait}s...")
                await asyncio.sleep(wait)
            else:
                print(f"  [Error {resp.status_code}] {ticker_label}")
                return None
        except Exception as e:
            await asyncio.sleep(1)
    return None

async def fetch_full_history(client, ticker, endpoint_url, semaphore):
    """Exhaustively paginate through every trade for a ticker."""
    all_trades = []
    cursor = None
    
    async with semaphore:
        while True:
            params = {"ticker": ticker, "limit": 1000}
            if cursor: params["cursor"] = cursor
            
            data = await fetch_with_backoff(client, endpoint_url, params, ticker)
            if not data: break
            
            batch = data.get('trades', [])
            for t in batch: t['ticker'] = ticker # Metadata injection
            all_trades.extend(batch)
            
            cursor = data.get('cursor')
            # Kalshi returns empty string or None when finished
            if not cursor or not batch:
                break
                
    return all_trades

async def run_exhaustive_extraction():
    # Use a limits object to prevent connection pool exhaustion
    limits = httpx.Limits(max_keepalive_connections=5, max_connections=10)
    async with httpx.AsyncClient(limits=limits) as client:
        semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
        
        # 1. DISCOVERY: Find every ticker via paginated market calls
        all_tickers = set()
        for event in EVENTS:
            for m_path in ["/markets", "/historical/markets"]:
                m_cursor = None
                while True:
                    m_params = {"event_ticker": event, "limit": 1000}
                    if m_cursor: m_params["cursor"] = m_cursor
                    
                    m_data = await fetch_with_backoff(client, f"{BASE_URL}{m_path}", m_params, f"Discovery-{event}")
                    if not m_data: break
                    
                    batch = m_data.get('markets', [])
                    all_tickers.update([m['ticker'] for m in batch])
                    m_cursor = m_data.get('cursor')
                    if not m_cursor or not batch: break

        print(f"Discovered {len(all_tickers)} unique tickers. Starting Trade Extraction...")

        # 2. EXTRACTION: Create tasks for BOTH endpoints for EVERY ticker
        # We use a dict to keep track of which ticker belongs to which result
        h_tasks = {ticker: fetch_full_history(client, ticker, f"{BASE_URL}/historical/trades", semaphore) for ticker in all_tickers}
        m_tasks = {ticker: fetch_full_history(client, ticker, f"{BASE_URL}/markets/trades", semaphore) for ticker in all_tickers}

        # Gather results
        print("Fetching from Historical Endpoint...")
        h_results = await tqdm.gather(*h_tasks.values(), desc="Historical API")
        
        print("Fetching from Markets Endpoint...")
        m_results = await tqdm.gather(*m_tasks.values(), desc="Markets API")

        # Flatten
        historical_final = [trade for sublist in h_results for trade in sublist]
        markets_final = [trade for sublist in m_results for trade in sublist]
        
        return historical_final, markets_final

historical_trades, markets_trades = await run_exhaustive_extraction()

Discovered 181 unique tickers. Starting Trade Extraction...
Fetching from Historical Endpoint...


Historical API: 100%|██████████| 181/181 [00:22<00:00,  8.19it/s]


Fetching from Markets Endpoint...


Markets API: 100%|██████████| 181/181 [00:34<00:00,  5.26it/s]


In [12]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

def save_to_endpoint_store(data, endpoint_name):
    """Writes a list of trade dicts to its specific sub-directory."""
    if not data:
        return
    
    base_path = f"data/market_data_store/{endpoint_name}"
    os.makedirs(base_path, exist_ok=True)
    
    df = pd.DataFrame(data)
    # Microstructure essentials: cast to high-precision types
    df['created_time'] = pd.to_datetime(df['created_time'])
    df['year'] = df['created_time'].dt.year
    
    # Write partitioned by year and ticker for fast lookups
    table = pa.Table.from_pandas(df)
    pq.write_to_dataset(
        table,
        root_path=base_path,
        partition_cols=['year', 'ticker'],
        compression='snappy'
    )
    print(f"Stored {len(df)} trades in {endpoint_name}")

# Example execution within your async loop
save_to_endpoint_store(historical_trades, "historical-endpoint")
save_to_endpoint_store(markets_trades, "markets-endpoint")

Stored 251221 trades in historical-endpoint
Stored 444946 trades in markets-endpoint


In [70]:
import pdfplumber
import re
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor # Changed to ThreadPool for Jupyter stability

# --- Functions remain the same ---

def extract_spatial_value(words, anchor_texts, x_tolerance=30, index=0, exclude_text=None):
    if isinstance(anchor_texts, str):
        anchor_texts = [anchor_texts]
    candidates = [w for w in words if any(t.upper() in w['text'].upper() for t in anchor_texts)]
    valid_anchors = []
    for cand in candidates:
        if exclude_text:
            preceded_by_excluded = any(
                exclude_text.upper() in w['text'].upper() 
                and abs(w['top'] - cand['top']) < 5
                and 0 < (cand['x0'] - w['x1']) < 80
                for w in words
            )
            if preceded_by_excluded: continue
        if "OPP." in cand['text'].upper() and "OPPONENT" in cand['text'].upper(): continue
        valid_anchors.append(cand)
    if not valid_anchors: return None
    valid_anchors.sort(key=lambda x: (x['top'], x['x0']))
    anchor = valid_anchors[0]
    column_digits = [w for w in words if abs(w['x0'] - anchor['x0']) < x_tolerance and w['top'] > anchor['bottom'] and re.match(r'^\d+$', w['text'])]
    column_digits.sort(key=lambda x: x['top'])
    return column_digits[index]['text'] if len(column_digits) > index else None

def get_header_metric(words, label_text):
    label_anchor = next((w for w in words if label_text.upper() in w['text'].upper()), None)
    if not label_anchor: return None
    candidates = [w for w in words if re.match(r'^\d+$', w['text']) and ((abs(w['top'] - label_anchor['top']) < 10 and w['x0'] > label_anchor['x1']) or (abs(w['x0'] - label_anchor['x0']) < 30 and w['top'] > label_anchor['bottom']))]
    candidates.sort(key=lambda x: (x['top'], x['x0']))
    return candidates[0]['text'] if candidates else None

def get_spatial_team_name(words, page_height):
    header_words = [w for w in words if w['top'] < page_height * 0.12 and not any(x in w['text'].upper() for x in ["THROUGH", "GAMES", "OFFICIAL", "PAGE"]) and not re.match(r'^\d{1,2}/\d{1,2}/\d{2,4}$', w['text'])]
    if not header_words: return None
    header_words.sort(key=lambda x: (x['top'], x['x0']))
    lines = []
    current_line = [header_words[0]]
    for i in range(1, len(header_words)):
        if abs(header_words[i]['top'] - header_words[i-1]['top']) < 3:
            current_line.append(header_words[i])
        else:
            lines.append(" ".join([w['text'] for w in current_line]))
            current_line = [header_words[i]]
    lines.append(" ".join([w['text'] for w in current_line]))
    clean = re.split(r'\(|:|Through Games|\(OFFICIAL\)', lines[0])[0].strip()
    if re.match(r"^(\w\s)+\w$", clean): clean = clean.replace(" ", "")
    return clean

def process_single_pdf(pdf_path):
    all_teams = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            full_text = " ".join([w['text'] for w in words])
            team_name = get_spatial_team_name(words, page.height)
            if not team_name: continue
            record_match = re.search(r"(\d{1,2}\s*-\s*\d{1,2})", full_text)
            metrics = {
                "Team": team_name,
                "Record": record_match.group(1).replace(" ", "") if record_match else None,
                "Avg_RPI_Win": get_header_metric(words, "Average RPI Win"),
                "Avg_RPI_Loss": get_header_metric(words, "Average RPI Loss"),
                "RPI_Rank_D1": extract_spatial_value(words, "TEAM", index=0),
                "RPI_Rank_NonConf": extract_spatial_value(words, "TEAM", index=1),
                "SOS_D1": extract_spatial_value(words, ["SUCCESS", "STRENGTH"], index=0, exclude_text="OPP"),
                "SOS_NonConf": extract_spatial_value(words, ["SUCCESS", "STRENGTH"], index=1, exclude_text="OPP"),
                "Opp_SOS_D1": extract_spatial_value(words, "OPP.", index=0),
                "Opp_SOS_NonConf": extract_spatial_value(words, "OPP.", index=1),
            }
            for key, pattern in [("NET", r"NET:\s*(\d+)"), ("KPI", r"KPI:\s*(\d+)"), ("SOR", r"SOR:\s*(\d+)"), ("POM", r"POM:\s*(\d+)"), ("SAG", r"SAG:\s*(\d+)"), ("BPI", r"BPI:\s*(\d+)")]:
                match = re.search(pattern, full_text)
                metrics[key] = match.group(1) if match else None
            all_teams.append(metrics)
    return os.path.splitext(os.path.basename(pdf_path))[0], all_teams

def clean_and_reconcile(year_key, raw_data, base_path):
    df = pd.DataFrame(raw_data)
    cols_to_check = [c for c in df.columns if c != 'Team']
    df = df.dropna(subset=cols_to_check, how='all').reset_index(drop=True)
    
    # Matching the exact naming convention for the reference file
    ref_path = os.path.join(base_path, "team_sheets", f"{year_key}_Team_Sheets_Final.csv")
    if not os.path.exists(ref_path):
        print(f"Warning: Reference CSV not found for {year_key} at {ref_path}")
        return df
    
    ref_df = pd.read_csv(ref_path)
    if len(df) != len(ref_df):
        print(f"Mismatch in {year_key}: Parsed {len(df)} teams, Reference has {len(ref_df)}.")
    
    # Map reference names to parsed data
    df['Team'] = ref_df.iloc[:, 0].values[:len(df)] 
    
    assert not df['Team'].duplicated().any(), f"Duplicate names in {year_key}"
    return df

# --- Main Execution Block for Jupyter ---

pdf_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/pdf/men/team_sheets/"
base_proj_path = "/Users/michaelharoon/Projects/tasty/march-madness/data/"
output_dir = "/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/"

os.makedirs(output_dir, exist_ok=True)
files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]

print(f"Processing {len(files)} files...")

# ThreadPoolExecutor is safe for Jupyter/macOS and still parallelizes well here
with ThreadPoolExecutor() as executor:
    results = list(executor.map(process_single_pdf, files))

for year_key, data in results:
    print(f"Cleaning {year_key}...")
    cleaned_df = clean_and_reconcile(year_key, data, base_proj_path)
    save_path = f"/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/{year_key}_Cleaned.csv"
    cleaned_df.to_csv(save_path, index=False)
    print(f"Saved: {save_path}")

Processing 15 files...
Cleaning 2008_Team_Sheets_Selection...
Saved: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2008_Team_Sheets_Selection_Cleaned.csv
Cleaning 2005_Team_Sheets_Final...
Saved: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2005_Team_Sheets_Final_Cleaned.csv
Cleaning 2015_Team_Sheets_Selection...
Saved: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2015_Team_Sheets_Selection_Cleaned.csv
Cleaning 2012_Team_Sheets_Selection...
Saved: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2012_Team_Sheets_Selection_Cleaned.csv
Cleaning 2007_Team_Sheets_Final...
Saved: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2007_Team_Sheets_Final_Cleaned.csv
Cleaning 2017_Team_Sheets_Selection...
Saved: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2017_Team_Sheets_Selection_Cleaned.csv
Cleaning 2018_Team_Sheets_Selection...


In [30]:
temp = final_data.copy()
temp

{'2008_Team_Sheets_Selection': [{'Team': 'A&M-Corpus Christi Of Saturday, March 15, 2008 SSSSSSSSS',
   'Record': '9-20',
   'Avg_RPI_Win': None,
   'Avg_RPI_Loss': None,
   'RPI_Rank_D1': '260',
   'RPI_Rank_NonConf': '265',
   'SOS_D1': '172',
   'SOS_NonConf': '222',
   'Opp_SOS_D1': '176',
   'Opp_SOS_NonConf': '107',
   'NET': None,
   'KPI': None,
   'SOR': None,
   'POM': None,
   'SAG': None},
  {'Team': 'Air Force Of Saturday, March 15, 2008',
   'Record': '14-14',
   'Avg_RPI_Win': None,
   'Avg_RPI_Loss': None,
   'RPI_Rank_D1': '167',
   'RPI_Rank_NonConf': '214',
   'SOS_D1': '146',
   'SOS_NonConf': '305',
   'Opp_SOS_D1': '146',
   'Opp_SOS_NonConf': '327',
   'NET': None,
   'KPI': None,
   'SOR': None,
   'POM': None,
   'SAG': None},
  {'Team': 'Akron Of Saturday, March 15, 2008',
   'Record': '23-10',
   'Avg_RPI_Win': None,
   'Avg_RPI_Loss': None,
   'RPI_Rank_D1': '76',
   'RPI_Rank_NonConf': '77',
   'SOS_D1': '134',
   'SOS_NonConf': '157',
   'Opp_SOS_D1': '148

In [31]:
import csv

def dict_to_csv(data_dict, output_directory="/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test"):
    # Create the directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    for filename, rows in data_dict.items():
        # Skip if there's no data for this file
        if not rows:
            continue
        
        # Build the full file path (adding .csv extension)
        file_path = os.path.join(output_directory, f"{filename}.csv")
        
        # Get the headers from the keys of the first dictionary in the list
        headers = rows[0].keys()
        
        with open(file_path, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            
            # Write the header row
            writer.writeheader()
            
            # Write all the data rows
            writer.writerows(rows)
            
        print(f"Created: {file_path}")

# Run the function
dict_to_csv(temp)

Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2008_Team_Sheets_Selection.csv
Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2005_Team_Sheets_Final.csv
Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2015_Team_Sheets_Selection.csv
Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2012_Team_Sheets_Selection.csv
Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2007_Team_Sheets_Final.csv
Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2017_Team_Sheets_Selection.csv
Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2018_Team_Sheets_Selection.csv
Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2010_Team_Sheets_Selection.csv
Created: /Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test/2011_Team_She

In [ ]:
import pandas as pd
import os
import re

def clean_team_name(raw_name):
    if not isinstance(raw_name, str):
        return raw_name

    # 1. Remove the 'SSSSSSSSS' marker
    name = raw_name.replace("SSSSSSSSS", "")

    # 2. Targeted "Of [Date/Status]" removal
    # This specifically looks for "Of" followed by a month, day, or 'Final'
    # It also handles surrounding commas.
    trash_of_patterns = r',?\s*\bOf\b\s+(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|March|Final|January|February|April)\b,?'
    name = re.sub(trash_of_patterns, '', name, flags=re.IGNORECASE)

    # 3. Targeted Metadata Parentheses removal
    # We only remove ( ) if they contain metric-related words or 'OFFICIAL'
    # This PROTECTS (Ill.), (Md.), (Fla.), etc.
    metadata_parens = r'\((?:RPI|NET|KPI|SOR|BPI|POM|SAG|AVG|OFFICIAL|Through).*?\)'
    name = re.sub(metadata_parens, '', name, flags=re.IGNORECASE)

    # 4. Remove Years (2005-2026)
    name = re.sub(r'\b(200[5-9]|201[0-9]|202[0-6])\b', '', name)

    # 5. Remove specific "Trash Words" that aren't part of names
    trash_words = r'\b(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|March|Final|OFFICIAL|Through|Games)\b'
    name = re.sub(trash_words, '', name, flags=re.IGNORECASE)

    # 6. Remove standalone day numbers (1-31)
    # This ignores numbers inside records (like 29-3) by ensuring no hyphens are attached
    name = re.sub(r'(?<!-)\b([1-9]|[12][0-9]|3[01])\b(?!-)', '', name)

    # 7. Remove trailing records and everything after (e.g., 29-3 NET)
    name = re.sub(r'\d{1,2}-\d{1,2}.*$', '', name)
    
    # 8. Clean up trailing Colons or stray "Of" leftovers
    name = name.strip()
    name = re.sub(r'[:]$', '', name)
    name = re.sub(r'\bOf\b$', '', name, flags=re.IGNORECASE)

    # 9. Final polish: whitespace and "K a n s a s" fix
    cleaned = re.sub(r'\s+', ' ', name).strip()
    if re.match(r"^(\w\s)+\w$", cleaned):
        cleaned = cleaned.replace(" ", "")

    return cleaned

# --- Batch Processing ---
base_path = '/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv'
test_path = os.path.join(base_path, 'test')

if not os.path.exists(test_path):
    os.makedirs(test_path)

for filename in os.listdir(base_path):
    if filename.lower().endswith(".csv"):
        df = pd.read_csv(os.path.join(base_path, filename))
        if 'Team' in df.columns:
            df['Team'] = df['Team'].apply(clean_team_name)
            df.to_csv(os.path.join(test_path, filename), index=False)
            print(f"Verified & Cleaned: {filename}")

Verified & Cleaned: 2006_Team_Sheets_Selection.csv
Verified & Cleaned: 2013_Team_Sheets_Selection.csv
Verified & Cleaned: 2014_Team_Sheets_Selection.csv
Verified & Cleaned: 2009_Team_Sheets_Selection.csv
Verified & Cleaned: 2011_Team_Sheets_Selection.csv
Verified & Cleaned: 2019_Team_Sheets_Selection.csv
Verified & Cleaned: 2016_Team_Sheets_Selection.csv
Verified & Cleaned: 2017_Team_Sheets_Selection.csv
Verified & Cleaned: 2018_Team_Sheets_Selection.csv
Verified & Cleaned: 2010_Team_Sheets_Selection.csv
Verified & Cleaned: 2008_Team_Sheets_Selection.csv
Verified & Cleaned: 2005_Team_Sheets_Final.csv
Verified & Cleaned: 2015_Team_Sheets_Selection.csv
Verified & Cleaned: 2012_Team_Sheets_Selection.csv
Verified & Cleaned: 2007_Team_Sheets_Final.csv


In [3]:
import pandas as pd
import os
import xlrd
from openpyxl import load_workbook
print(xlrd.__version__)

def get_unique_columns(directory_path):
    if not os.path.exists(directory_path):
        print(f"Error: The directory {directory_path} does not exist.")
        return set()

    unique_columns = set()
    file_count = 0

    # Iterate through the year-based folders
    for i in range(2003, 2027):
        parent_path = os.path.join(directory_path, f"{i}-team-stats")
        
        # Check if the year folder actually exists before listing
        if not os.path.exists(parent_path):
            continue

        for filename in os.listdir(parent_path):
            file_path = os.path.join(parent_path, filename)
            
            if os.path.isfile(file_path):
                try:
                    # Handle CSV files
                    if filename.endswith('.csv'):
                        df = pd.read_csv(file_path, nrows=0)
                        unique_columns.update(df.columns.tolist())
                        file_count += 1
                    

                    elif filename.endswith('.xls'):
                        try:
                            # Try as true XLS
                            df = pd.read_excel(file_path, engine='xlrd', nrows=0)
                        except Exception:
                            try:
                                # Force openpyxl in read-only mode (bypasses style issues)
                                wb = load_workbook(file_path, read_only=True, data_only=True)
                                ws = wb.active

                                # Extract header row manually
                                header = [cell.value for cell in next(ws.iter_rows(max_row=1))]
                                df = pd.DataFrame(columns=header)

                            except Exception as e:
                                print(f"Could not read {filename}: {e}")
                                continue

                        unique_columns.update(df.columns.tolist())
                        file_count += 1

                except Exception as e:
                    print(f"Could not read {filename}: {e}")

    print(f"Processed {file_count} files (CSV and XLS).")
    return sorted(list(unique_columns))

# --- Execution ---
target_folder = '/Users/michaelharoon/Projects/tasty/march-madness/data'
all_cols = get_unique_columns(target_folder)

print(f"{len(all_cols)} Unique Column Names Found:")
print(60 * "-")
print(*all_cols, sep=",")

2.0.1
Processed 393 files (CSV and XLS).
51 Unique Column Names Found:
------------------------------------------------------------
3FG,3FG%,3FGA,3PG,APG,AST,Avg,BKPG,BLKS,Bench,DQ,DRebs,FB pts,FG%,FGA,FGM,FT,FT%,FTA,Fouls,G,GM,L,OPP FG,OPP FG%,OPP FGA,OPP PPG,OPP PTS,OPP REB,OPP RPG,ORebs,Opp 3FG,Opp 3FGA,Opp TO,PFPG,PPG,PTS,Pct,REB,REB MAR,RPG,Rank,Ratio,SCR MAR,ST,STPG,TO,TOPG,Team,W,W-L


In [4]:
import pathlib
import pyarrow.parquet as pq

base_path = '/Users/michaelharoon/Projects/tasty/march-madness/data/market_data_store'
all_columns = set()

# Recursively find all parquet files
for path in pathlib.Path(base_path).rglob('*.parquet'):
    try:
        # Read only the file metadata/schema
        schema = pq.read_schema(path)
        all_columns.update(schema.names)
    except Exception as e:
        print(f"Could not read {path}: {e}")

print("Distinct Columns:", sorted(list(all_columns)))


Distinct Columns: ['count_fp', 'created_time', 'no_price_dollars', 'taker_side', 'trade_id', 'yes_price_dollars']


In [35]:
import pandas as pd
import os

def calculate_win_pct(record):
    """Parses 'W-L' string and returns W / (W + L)."""
    try:
        if pd.isna(record) or '-' not in str(record):
            return 0
        wins, losses = map(int, str(record).split('-'))
        if (wins + losses) == 0:
            return 0
        return wins / (wins + losses)
    except (ValueError, ZeroDivisionError):
        return 0

directory_path = '/Users/michaelharoon/Projects/tasty/march-madness/data'

# Process only CSV files in the first level of the directory
for filename in os.listdir(directory_path):
    if filename.endswith(".csv") and filename.startswith("20"):
        file_path = os.path.join(directory_path, filename)
        df = pd.read_csv(file_path)

        df.rename(
            columns={
                'RB_KPI': 'KPI',
                'RB_SOR': 'SOR',
                'PM_BPI': "BPI",
                'PM_POM': "POM",
                'PM_SAG': "SAG"
            },
            inplace=True)

        # Save the updated CSV
        df.to_csv(file_path, index=False)


In [112]:
import pandas as pd
import os

directory_path = '/Users/michaelharoon/Projects/tasty/march-madness/data/raw/dirty_csv/test'

for filename in os.listdir(directory_path):
    if filename.endswith(".csv") and filename.startswith("20"):
        file_path = os.path.join(directory_path, filename)
        
        try:
            df = pd.read_csv(file_path)
            
            if 'Team' in df.columns:
                # 1. Clean the team names for comparison (strip spaces and uppercase)
                # This ensures "Duke" and "duke " are caught as duplicates
                clean_names = df['Team'].astype(str).str.strip().str.upper()
                
                # 2. Find all instances of duplicates
                # keep=False marks every occurrence of a duplicate as True
                is_duplicate = clean_names.duplicated(keep=False)
                
                if is_duplicate.any():
                    # 3. Filter the original dataframe to show the dupes
                    dupes_found = df[is_duplicate].sort_values(by='Team')
                    
                    print(f"\n[!] {filename}: Found {len(dupes_found)} rows with duplicate team names:")
                    print("-" * 50)
                    # Showing 'Team' and any other relevant columns you might want to see
                    print(dupes_found[['Team']]) 
                    print("-" * 50)
                else:
                    print(f"Clean: {filename}")
                    
        except Exception as e:
            print(f"Error reading {filename}: {e}")

Clean: 2006_Team_Sheets_Selection.csv
Clean: 2013_Team_Sheets_Selection.csv
Clean: 2014_Team_Sheets_Selection.csv
Clean: 2009_Team_Sheets_Selection.csv
Clean: 2011_Team_Sheets_Selection.csv
Clean: 2019_Team_Sheets_Selection.csv
Clean: 2016_Team_Sheets_Selection.csv
Clean: 2017_Team_Sheets_Selection.csv
Clean: 2018_Team_Sheets_Selection.csv
Clean: 2010_Team_Sheets_Selection.csv
Clean: 2008_Team_Sheets_Selection.csv
Clean: 2005_Team_Sheets_Final.csv
Clean: 2015_Team_Sheets_Selection.csv
Clean: 2012_Team_Sheets_Selection.csv
Clean: 2007_Team_Sheets_Final.csv


In [66]:
import pandas as pd
ranks = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')
ranks[ranks['SystemName']=='BPI']

,Season,RankingDayNum,SystemName,TeamID,OrdinalRank
880139,2009,57,BPI,1102,161
880140,2009,57,BPI,1103,96
880141,2009,57,BPI,1104,105
880142,2009,57,BPI,1105,329
880143,2009,57,BPI,1106,299
...,...,...,...,...,...
2002469,2013,133,BPI,1460,126
2002470,2013,133,BPI,1461,70
2002471,2013,133,BPI,1462,72
2002472,2013,133,BPI,1463,217


In [65]:
import pandas as pd

# 1. Load Data
ranks = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/kaggle/MMasseyOrdinals.csv')
ts_2026 = pd.read_csv('/Users/michaelharoon/Projects/tasty/march-madness/data/team_sheets/2026_Team_Sheet_Final.csv')

# Standardize Massey naming
system_col = 'SystemName' if 'SystemName' in ranks.columns else 'SysteName'

# 2. Identify Massey Historical vs 2026
historical_systems = set(ranks[ranks['Season'] < 2026][system_col].unique())
systems_2026_massey = set(ranks[ranks['Season'] == 2026][system_col].unique())

# Systems that existed before but are missing from 2026 Massey data
dropped_in_2026 = historical_systems - systems_2026_massey

# 3. Analyze 2026 Team Sheet coverage
# Identify columns in Team Sheet that have at least one non-null value
ts_available = [col for col in ts_2026.columns if ts_2026[col].notna().any()]
ts_available_set = set(ts_available)

# Mapping dictionary for known aliases (Team Sheet vs Massey abbreviations)
# Standard Massey codes: SAG (Sagarin), POM (Pomeroy), BPI (ESPN), MOR (Moore), etc.
mapping = {
    'POM': 'POM',
    'SAG': 'SAG',
    'BPI': 'BPI',
    'KPI': 'KPI',
    'SOR': 'SOR',
    'NET_Rank': 'NET',
    'RPI_Rank_D1': 'RPI',
    'PM_T-Rank': 'TRK' # BartTorvik
}

# 4. Logical Comparison
can_fill_dropped = [m for m in dropped_in_2026 if m in ts_available_set or m in mapping.values()]
new_to_2026_ts = ts_available_set - historical_systems

# --- REPORTING ---
print(f"--- MASSEY 2026 GAP ANALYSIS ---")
print(f"Systems active in previous years: {len(historical_systems)}")
print(f"Systems active in 2026 Massey:    {len(systems_2026_massey)}")
print(f"Missing from 2026 Massey:        {len(dropped_in_2026)}")

print(f"\n--- TEAM SHEET RECOVERY POTENTIAL ---")
if can_fill_dropped:
    print(f"Team Sheet can RECOVER these missing Massey systems: {can_fill_dropped}")
else:
    print("Team Sheet does not contain any of the dropped Massey systems.")

print(f"\nNew/Custom features in Team Sheet (not in historical Massey):")
print([c for c in new_to_2026_ts if c not in mapping])

# Detailed check for your "Big Three"
for sys in ['SAG', 'POM', 'BPI']:
    in_massey = sys in systems_2026_massey
    in_ts = sys in ts_available_set
    status = "AVAILABLE" if in_ts or in_massey else "MISSING EVERYWHERE"
    print(f"{sys:3} Status: {status} (Massey: {in_massey}, TeamSheet: {in_ts})")

--- MASSEY 2026 GAP ANALYSIS ---
Systems active in previous years: 193
Systems active in 2026 Massey:    60
Missing from 2026 Massey:        137

--- TEAM SHEET RECOVERY POTENTIAL ---
Team Sheet can RECOVER these missing Massey systems: ['SAG', 'BPI', 'KPI']

New/Custom features in Team Sheet (not in historical Massey):
['Conference_Record', 'Overall_Record', 'RB_WAB', 'NET_NonConf_SOS', 'Road_Record', 'Team', 'RPI_Rank_NonConf', 'Avg_NET_Wins', 'NET_SOS', 'Avg_NET_Losses', 'NonConf_Record']
SAG Status: MISSING EVERYWHERE (Massey: False, TeamSheet: False)
POM Status: AVAILABLE (Massey: True, TeamSheet: True)
BPI Status: AVAILABLE (Massey: False, TeamSheet: True)


In [106]:
systems = df['SystemName'].unique()
systems

<ArrowStringArray>
['SEL',  'AP', 'BIH', 'DUN', 'ENT', 'GRN', 'IMS', 'MAS', 'MKV', 'MOR',
 ...
 'RMS', 'RME', 'STY', 'PAC', 'BMN', 'WAB', 'WEI', 'ENG', 'AEI', 'TBL']
Length: 197, dtype: str